
# Analyse exploratoire des données FAO

Dans ce notebook, nous allons analyser les données alimentaires de la FAO afin d'identifier les principales tendances entre 1961 et 2013.

L'analyse portera sur les quantités destinées à l'alimentation humaine (`Food`) et animale (`Feed`), ainsi que sur les différences entre les pays et les produits alimentaires.

L'objectif est d'obtenir des résultats pertinents qui pourront être utilisés dans le Data Storytelling de l'application Dash.

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:

# Charger le dataset nettoyé
df = pd.read_csv("../data/FAO_clean.csv")

# Afficher les dimensions
print("Dimensions :", df.shape)

# Afficher les premières lignes
df.head()

Dimensions : (1138281, 12)


,Area Abbreviation,Area Code,Area,Item Code,Item,Element Code,Element,Unit,latitude,longitude,Year,Value
0,AFG,2,Afghanistan,2511,Wheat and products,5142,Food,1000 tonnes,33.94,67.71,1961,1928.0
1,AFG,2,Afghanistan,2805,Rice (Milled Equivalent),5142,Food,1000 tonnes,33.94,67.71,1961,183.0
2,AFG,2,Afghanistan,2513,Barley and products,5521,Feed,1000 tonnes,33.94,67.71,1961,76.0
3,AFG,2,Afghanistan,2513,Barley and products,5142,Food,1000 tonnes,33.94,67.71,1961,237.0
4,AFG,2,Afghanistan,2514,Maize and products,5521,Feed,1000 tonnes,33.94,67.71,1961,210.0



## 1. Analyse statistique descriptive

Nous commençons par calculer les principales statistiques descriptives des quantités alimentaires afin de mieux comprendre leur distribution.

Cette analyse permettra d'identifier les ordres de grandeur, la dispersion des valeurs et les éventuelles différences entre l'alimentation humaine et animale.


In [3]:

# Statistiques descriptives des quantités alimentaires
df["Value"].describe()


count    1.020829e+06
mean     3.709970e+02
std      3.832204e+03
min      0.000000e+00
25%      0.000000e+00
50%      3.000000e+00
75%      4.900000e+01
max      4.892990e+05
Name: Value, dtype: float64


### Observations sur les statistiques descriptives

L'analyse porte sur **1 020 829 valeurs disponibles**.

La quantité moyenne est d'environ **371 milliers de tonnes**, tandis que la médiane est de seulement **3 milliers de tonnes**.

Cet écart indique une distribution fortement asymétrique, avec quelques quantités très élevées.

L'écart-type de **3 832** confirme une forte dispersion des valeurs.

Le premier quartile est nul et la valeur maximale atteint **489 299 milliers de tonnes**.

Ces résultats montrent que les quantités alimentaires sont très inégalement réparties entre les observations. Il sera donc important de distinguer les pays, les produits et les usages alimentaires dans la suite de l'analyse.


In [4]:

# Statistiques descriptives par type d'utilisation
stats_element = df.groupby("Element")["Value"].describe()

stats_element.round(2)

,count,mean,std,min,25%,50%,75%,max
Element,,,,,,,,
Feed,181703.0,480.72,4159.95,0.0,0.0,3.0,61.0,193043.0
Food,839126.0,347.24,3757.05,0.0,0.0,4.0,47.0,489299.0



### Observations sur les catégories Food et Feed

L'analyse montre que la quantité moyenne par observation est plus élevée pour **Feed (480,72)** que pour **Food (347,24)**, en milliers de tonnes.

Cependant, les médianes sont proches : **3 pour Feed** et **4 pour Food**.

Les deux catégories présentent une forte dispersion, avec des écarts-types nettement supérieurs aux moyennes.

Le nombre d'observations est également très différent : **839 126 pour Food**, contre **181 703 pour Feed**.

Ces résultats montrent que les distributions sont asymétriques. La comparaison des moyennes ne suffit donc pas pour déterminer la répartition globale des quantités alimentaires.

In [5]:

# Calculer les quantités totales par utilisation
volumes_element = df.groupby("Element")["Value"].sum()

# Calculer leur part dans le total
parts_element = (
    volumes_element / volumes_element.sum() * 100
).round(2)

# Créer un tableau récapitulatif
comparaison_element = pd.DataFrame({
    "Volume total (1000 tonnes)": volumes_element,
    "Part du total (%)": parts_element
})

comparaison_element

,Volume total (1000 tonnes),Part du total (%)
Element,,
Feed,87348711.0,23.06
Food,291375778.0,76.94


In [6]:

# Calculer les volumes annuels par utilisation
volumes_annuels = (
    df.groupby(["Year", "Element"])["Value"]
    .sum()
    .unstack()
)

# Afficher les premières années
volumes_annuels.head(10)


Element,Feed,Food
Year,,
1961,884314.0,2618297.0
1962,913198.0,2688434.0
1963,954852.0,2730772.0
1964,972294.0,2793351.0
1965,1006792.0,2895741.0
1966,1062976.0,2990814.0
1967,1080316.0,3052944.0
1968,1145891.0,3130806.0
1969,1163298.0,3219690.0


In [7]:

import plotly.express as px

# Préparer les données pour la visualisation
data_plot = comparaison_element.reset_index()

# Créer le graphique
fig = px.bar(
    data_plot,
    x="Element",
    y="Volume total (1000 tonnes)",
    color="Element",
    title="Répartition des volumes alimentaires cumulés (1961-2013)",
    labels={
        "Element": "Type d'utilisation",
        "Volume total (1000 tonnes)": "Volume cumulé (milliers de tonnes)"
    },
    text="Part du total (%)",
    color_discrete_map={
        "Food": "#238C88",
        "Feed": "#E6A24A"
    }
)

fig.update_traces(texttemplate="%{text:.2f} %", textposition="outside")
fig.update_layout(showlegend=False)

fig.show()



### Observations sur les volumes cumulés

Sur la période 1961–2013, les quantités alimentaires cumulées se répartissent comme suit :

- **Food : 76,94 %**, soit 291 375 778 milliers de tonnes.
- **Feed : 23,06 %**, soit 87 348 711 milliers de tonnes.

Les volumes cumulés destinés à l'alimentation humaine sont donc environ **3,34 fois supérieurs** à ceux destinés à l'alimentation animale.

Ces proportions concernent uniquement les données disponibles dans notre dataset et l'ensemble de la période étudiée.

Une analyse annuelle sera nécessaire pour comprendre comment cette répartition a évolué dans le temps.



## 2. Évolution annuelle des usages alimentaires

Nous allons analyser l'évolution de la répartition des quantités alimentaires entre `Food` et `Feed` de 1961 à 2013.

L'objectif est d'observer si la part destinée à l'alimentation humaine et animale évolue au cours du temps.


In [8]:

# Calculer les volumes par année et par usage
volumes_annuels = (
    df.groupby(["Year", "Element"])["Value"]
    .sum()
    .unstack()
)

# Calculer les proportions annuelles en pourcentage
proportions_annuelles = (
    volumes_annuels.div(volumes_annuels.sum(axis=1), axis=0) * 100
)

# Afficher les premières années
proportions_annuelles.head(10).round(2)


Element,Feed,Food
Year,,
1961,25.25,74.75
1962,25.36,74.64
1963,25.91,74.09
1964,25.82,74.18
1965,25.80,74.20
1966,26.22,73.78
1967,26.14,73.86
1968,26.79,73.21
1969,26.54,73.46


In [9]:

# Transformer les proportions pour Plotly
data_proportions = proportions_annuelles.reset_index().melt(
    id_vars="Year",
    value_vars=["Food", "Feed"],
    var_name="Usage",
    value_name="Proportion"
)

# Créer le graphique
fig = px.line(
    data_proportions,
    x="Year",
    y="Proportion",
    color="Usage",
    title="Évolution des proportions alimentaires (1961-2013)",
    labels={
        "Year": "Année",
        "Proportion": "Part des quantités (%)",
        "Usage": "Utilisation"
    },
    color_discrete_map={
        "Food": "#238C88",
        "Feed": "#E6A24A"
    }
)

fig.update_layout(
    hovermode="x unified",
    yaxis=dict(range=[0, 100])
)

fig.show()



### Observations sur l'évolution annuelle des proportions

Le graphique montre que l'alimentation humaine (`Food`) reste majoritaire sur l'ensemble de la période 1961–2013.

Sa proportion augmente progressivement, passant d'environ **75 % à près de 80 %**, tandis que celle de l'alimentation animale (`Feed`) diminue d'environ **25 % à 20 %**.

Les variations restent relativement faibles, malgré quelques fluctuations au cours de la période.

Cette évolution concerne les proportions et ne signifie pas nécessairement que les volumes destinés à l'alimentation animale ont diminué.

Les différences de disponibilité des données selon les années doivent également être prises en compte dans l'interprétation.



## 3. Évolution des volumes alimentaires absolus

Après avoir étudié les proportions annuelles de `Food` et `Feed`, nous allons analyser l'évolution des quantités absolues.

L'objectif est de déterminer si les volumes alimentaires ont augmenté ou diminué entre 1961 et 2013, indépendamment de leur part relative.


In [10]:

# Préparer les volumes annuels pour Plotly
data_volumes = volumes_annuels.reset_index().melt(
    id_vars="Year",
    value_vars=["Food", "Feed"],
    var_name="Usage",
    value_name="Volume"
)

# Créer le graphique
fig = px.line(
    data_volumes,
    x="Year",
    y="Volume",
    color="Usage",
    title="Évolution des volumes alimentaires annuels (1961-2013)",
    labels={
        "Year": "Année",
        "Volume": "Volume annuel (milliers de tonnes)",
        "Usage": "Utilisation"
    },
    color_discrete_map={
        "Food": "#238C88",
        "Feed": "#E6A24A"
    }
)

fig.update_layout(
    hovermode="x unified",
    yaxis=dict(rangemode="tozero")
)

fig.show()



### Observations sur l'évolution des volumes absolus

Le graphique montre une augmentation des volumes alimentaires enregistrés entre 1961 et 2013.

Les quantités destinées à l'alimentation humaine (`Food`) progressent plus fortement que celles destinées à l'alimentation animale (`Feed`).

Bien que la proportion de `Feed` diminue sur cette période, son volume absolu augmente. Cela montre l'importance de distinguer les quantités et les proportions.

Une hausse plus marquée apparaît autour de 1992. Elle pourrait être partiellement liée à l'amélioration de la couverture des données, observée lors du diagnostic des valeurs manquantes.

Ces résultats doivent donc être interprétés avec prudence.



## 4. Analyse des produits alimentaires

Nous allons identifier les produits alimentaires qui représentent les plus grandes quantités dans le dataset.

L'objectif est de comprendre quels produits occupent une place importante dans les volumes enregistrés et d'étudier leur répartition entre l'alimentation humaine (`Food`) et animale (`Feed`).


In [11]:

# Calculer les volumes cumulés par produit
top_produits = (
    df.groupby("Item")["Value"]
    .sum()
    .nlargest(10)
    .reset_index()
)

# Afficher les résultats
top_produits


,Item,Value
0,Cereals - Excluding Beer,64884281.0
1,Milk - Excluding Butter,45014120.0
2,Vegetables,24179916.0
3,Starchy Roots,22711529.0
4,Maize and products,19960640.0
5,Wheat and products,19194671.0
6,"Vegetables, Other",19052945.0
7,Rice (Milled Equivalent),14475448.0
8,Fruits - Excluding Wine,14420114.0
9,Potatoes and products,9545562.0


In [12]:

# Créer un graphique horizontal des 10 principaux produits
fig = px.bar(
    top_produits.sort_values("Value"),
    x="Value",
    y="Item",
    orientation="h",
    title="Top 10 des catégories alimentaires par volume cumulé (1961-2013)",
    labels={
        "Value": "Volume cumulé (milliers de tonnes)",
        "Item": "Produit alimentaire"
    },
    color_discrete_sequence=["#238C88"]
)

fig.update_layout(
    yaxis_title="",
    showlegend=False
)

fig.show()



### Observations sur les principaux produits alimentaires

Le graphique montre que les céréales hors bière représentent le volume cumulé le plus important entre 1961 et 2013, suivies du lait hors beurre et des légumes.

Le maïs, le blé et le riz figurent également parmi les produits les plus représentés en quantité.

Cependant, le classement contient à la fois des catégories générales et des produits spécifiques, ce qui peut entraîner des chevauchements.

Ces résultats donnent un premier aperçu des produits dominants dans le dataset, mais doivent être interprétés avec prudence.



## 5. Répartition des usages alimentaires par produit

Nous allons comparer les quantités destinées à l'alimentation humaine (`Food`) et animale (`Feed`) pour les dix principales catégories alimentaires.

L'objectif est d'identifier les différences d'utilisation selon les produits.


In [13]:

# Sélectionner les 10 principales catégories
produits_selectionnes = top_produits["Item"]

# Calculer les volumes par produit et par usage
repartition_produits = (
    df[df["Item"].isin(produits_selectionnes)]
    .groupby(["Item", "Element"])["Value"]
    .sum()
    .reset_index()
)

# Créer le graphique
fig = px.bar(
    repartition_produits,
    x="Value",
    y="Item",
    color="Element",
    orientation="h",
    barmode="stack",
    title="Répartition Food et Feed des principaux produits (1961-2013)",
    labels={
        "Value": "Volume cumulé (milliers de tonnes)",
        "Item": "Produit alimentaire",
        "Element": "Utilisation"
    },
    color_discrete_map={
        "Food": "#238C88",
        "Feed": "#E6A24A"
    },
    category_orders={
        "Item": top_produits["Item"].iloc[::-1].tolist()
    }
)

fig.update_layout(
    yaxis_title="",
    hovermode="closest"
)

fig.show()



### Observations sur la répartition Food et Feed par produit

Le graphique montre que l'utilisation des produits alimentaires varie selon leur nature.

Les céréales hors bière présentent les volumes cumulés les plus importants, avec une répartition entre alimentation humaine et animale.

Le maïs est majoritairement associé à l'alimentation animale (`Feed`), tandis que les légumes, les fruits et le riz sont principalement destinés à l'alimentation humaine (`Food`).

Ces résultats mettent en évidence des différences importantes selon les produits.

Cependant, certaines catégories sont agrégées et peuvent se chevaucher avec des produits plus spécifiques. Les volumes doivent donc être interprétés avec prudence.



## 6. Analyse géographique des volumes alimentaires

Nous allons comparer les quantités alimentaires enregistrées dans les différents pays du dataset FAO.

L'objectif est d'identifier les pays présentant les volumes les plus importants et d'étudier les différences entre l'alimentation humaine (`Food`) et animale (`Feed`).


In [14]:

# Catégories agrégées à exclure provisoirement
categories_agregees = [
    "Cereals - Excluding Beer",
    "Vegetables",
    "Fruits - Excluding Wine",
    "Starchy Roots",
    "Milk - Excluding Butter"
]

# Créer un sous-ensemble pour les comparaisons géographiques
df_pays = df[~df["Item"].isin(categories_agregees)].copy()

# Calculer les volumes cumulés par pays et par usage
volumes_pays = (
    df_pays.groupby(["Area", "Element"])["Value"]
    .sum()
    .unstack(fill_value=0)
)

# Classer les pays selon leurs volumes cumulés
top_10_pays = (
    volumes_pays
    .assign(Total=lambda x: x["Food"] + x["Feed"])
    .nlargest(10, "Total")
)

top_10_pays


Element,Feed,Food,Total
Area,,,
"China, mainland",8392449.0,35468906.0,43861355.0
United States of America,7967945.0,14516285.0,22484230.0
India,905608.0,18045500.0,18951108.0
Brazil,2542757.0,5538570.0,8081327.0
Germany,1891471.0,4600549.0,6492020.0
Japan,1126576.0,4982771.0,6109347.0
France,1221419.0,3261230.0,4482649.0
Indonesia,169388.0,4149125.0,4318513.0
Nigeria,846005.0,3448427.0,4294432.0


In [15]:

# Préparer les données pour Plotly
data_pays = (
    top_10_pays[["Food", "Feed"]]
    .reset_index()
    .melt(
        id_vars="Area",
        var_name="Usage",
        value_name="Volume"
    )
)

# Conserver l'ordre du classement
ordre_pays = top_10_pays.index[::-1].tolist()

# Créer le graphique à points
fig = px.scatter(
    data_pays,
    x="Volume",
    y="Area",
    color="Usage",
    title="Comparaison des volumes Food et Feed par pays (1961-2013)",
    labels={
        "Volume": "Volume cumulé (milliers de tonnes)",
        "Area": "Pays",
        "Usage": "Utilisation"
    },
    color_discrete_map={
        "Food": "#238C88",
        "Feed": "#E6A24A"
    },
    category_orders={"Area": ordre_pays}
)

fig.update_traces(marker=dict(size=12))

fig.update_layout(
    yaxis_title="",
    hovermode="closest"
)

fig.show()



### Observations sur les volumes alimentaires par pays

Le graphique montre que la Chine continentale présente les volumes cumulés les plus importants parmi les dix pays étudiés, suivie des États-Unis et de l'Inde.

Dans tous les pays sélectionnés, les quantités destinées à l'alimentation humaine (`Food`) sont supérieures à celles destinées à l'alimentation animale (`Feed`).

Les États-Unis se distinguent par un volume important d'alimentation animale, tandis que l'Inde présente un écart particulièrement marqué en faveur de l'alimentation humaine.

Ces résultats concernent les volumes cumulés disponibles entre 1961 et 2013, après l'exclusion provisoire de certaines catégories agrégées. Ils doivent être interprétés avec prudence.



## 7. Répartition géographique des volumes alimentaires

Nous allons représenter les volumes alimentaires sur une carte mondiale interactive.

L'objectif est de visualiser les différences géographiques entre les pays et de pouvoir sélectionner les quantités destinées à l'alimentation humaine (`Food`) ou animale (`Feed`).


In [16]:

# Regrouper les volumes par pays et par usage
data_carte = (
    df_pays.groupby(
        ["Area", "latitude", "longitude", "Element"],
        as_index=False
    )["Value"]
    .sum()
)

# Renommer les colonnes pour faciliter la lecture
data_carte = data_carte.rename(columns={
    "Element": "Usage",
    "Value": "Volume"
})

# Vérifier les premières lignes
data_carte.head()


,Area,latitude,longitude,Usage,Volume
0,Afghanistan,33.94,67.71,Feed,19339.0
1,Afghanistan,33.94,67.71,Food,299378.0
2,Albania,41.15,20.17,Feed,13239.0
3,Albania,41.15,20.17,Food,89928.0
4,Algeria,28.03,1.66,Feed,66787.0


In [17]:

import plotly.graph_objects as go

# Couleurs des deux catégories
couleurs = {
    "Food": "#238C88",
    "Feed": "#E6A24A"
}

fig = go.Figure()

# Ajouter une couche de bulles pour chaque utilisation
for usage in ["Food", "Feed"]:
    donnees = data_carte[data_carte["Usage"] == usage]

    fig.add_trace(
        go.Scattergeo(
            lat=donnees["latitude"],
            lon=donnees["longitude"],
            text=donnees["Area"],
            customdata=donnees["Volume"],
            mode="markers",
            name=usage,
            visible=(usage == "Food"),
            marker=dict(
                size=donnees["Volume"].pow(0.5) / 45,
                sizemin=4,
                color=couleurs[usage],
                opacity=0.7,
                line=dict(width=0.5, color="white")
            ),
            hovertemplate=(
                "<b>%{text}</b>"
                "<br>Volume : %{customdata:,.0f} milliers de tonnes"
                "<extra></extra>"
            )
        )
    )

# Agrandir la carte et ajouter les boutons
fig.update_layout(
    title="Répartition mondiale des volumes alimentaires (1961-2013)",
    width=1200,
    height=700,
    margin=dict(l=20, r=20, t=100, b=20),

    geo=dict(
        projection_type="natural earth",
        showland=True,
        landcolor="#F0F2F5",
        showocean=True,
        oceancolor="#E6F1F5",
        showcountries=True,
        countrycolor="#A0A0A0"
    ),

    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.5,
            xanchor="center",
            y=1.08,
            buttons=[
                dict(
                    label="Food",
                    method="update",
                    args=[{"visible": [True, False]}]
                ),
                dict(
                    label="Feed",
                    method="update",
                    args=[{"visible": [False, True]}]
                )
            ]
        )
    ],

    showlegend=False
)

fig.show()



### Observations sur la répartition géographique

La carte interactive met en évidence des différences importantes entre les volumes alimentaires des pays.

Les volumes cumulés les plus élevés se concentrent notamment dans de grands pays comme la Chine, les États-Unis et l'Inde.

La sélection entre `Food` et `Feed` permet de constater que la répartition géographique varie selon l'utilisation alimentaire.

Ces résultats concernent les quantités cumulées entre 1961 et 2013, après l'exclusion provisoire de certaines catégories agrégées.

Les volumes ne sont pas rapportés à la population et ne permettent donc pas de comparer la consommation alimentaire par habitant.



## 8. Conclusion de l'analyse exploratoire

Cette analyse exploratoire nous a permis de mieux comprendre la répartition et l'évolution des quantités alimentaires enregistrées dans le dataset FAO entre 1961 et 2013.

### Principaux résultats

Plusieurs observations ressortent de cette analyse :

- **Répartition Food / Feed :** les quantités cumulées destinées à l'alimentation humaine représentent 76,94 % du total étudié, contre 23,06 % pour l'alimentation animale.

- **Évolution temporelle :** les volumes enregistrés augmentent globalement entre 1961 et 2013. La part relative de `Food` progresse, tandis que celle de `Feed` diminue, même si ses volumes absolus augmentent.

- **Produits alimentaires :** les céréales hors bière, le lait hors beurre et les légumes figurent parmi les catégories présentant les volumes cumulés les plus importants. Le maïs se distingue par une utilisation majoritairement destinée à l'alimentation animale.

- **Répartition géographique :** la Chine continentale, les États-Unis et l'Inde figurent parmi les pays enregistrant les volumes les plus élevés dans notre classement exploratoire.

### Limites de l'analyse

Certaines limites doivent être prises en compte :

- Les données manquantes sont plus nombreuses au début de la période étudiée.
- Les volumes cumulés ne tiennent pas compte des différences de population entre les pays.
- Certaines catégories alimentaires sont agrégées et peuvent se chevaucher avec des produits plus détaillés.
- Les quantités étudiées ne représentent pas directement la consommation réelle des populations.

Ces limites nécessitent une interprétation prudente des résultats.

### Perspectives pour Dazzle Dash

Cette analyse nous a permis d'identifier plusieurs informations intéressantes à présenter dans mon application Dash.

Nous souhaitons notamment mettre en avant l'évolution des usages alimentaires, les différences entre les produits et les comparaisons géographiques.

L'objectif sera de sélectionner les visualisations les plus pertinentes et de les accompagner d'explications simples afin de rendre les résultats accessibles à un public non technique.
